In [39]:
# 0myStrategy/Covered Call.ipynb
# -*- coding: utf-8 -*-

import sys
import os
from pathlib import Path
notebook_path = os.path.abspath('')  # مسیر فعلی
root_dir = Path(notebook_path).parent
sys.path.append(str(root_dir))

from data.downloader import MarketDownloader
from data.cleaner import DataCleaner
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime
import random
from scipy.stats import norm

df_raw =  MarketDownloader.from_tsetmc_direct()
df_cleaned = DataCleaner.clean(df_raw)
df_final = DataCleaner.add_derived_columns(df_cleaned)
df_final.head(3)


,Ticker,Name,StrikePrice,UnderlyingTicker,UnderlyingPrice,MaturityDate,DaysToMaturity,OpenPositions,Volume,Value,...,AskVolume,InstrumentCode,InstrumentCode-UA,IntrinsicValue,MidPrice,TimeValue,Moneyness,OptionStatus,SpreadPct,PremiumOverIntrinsic
0,ضهرم4023,اختيارخ اهرم-20000-1405/04/31,20000,اهرم,39488,1405-04-31,6,94722,794,3.179494e+10,...,20,3729523725493580,17914401175772326,19488.0,19200.5,0.0,1.974400,ITM,0.020781,0.989635
1,ضهرم4024,اختيارخ اهرم-22000-1405/04/31,22000,اهرم,39488,1405-04-31,6,38666,802,3.211529e+10,...,2,58652628290274333,17914401175772326,17488.0,9011.0,0.0,1.794909,ITM,1.994895,0.972667
2,ضهرم4025,اختيارخ اهرم-24000-1405/04/31,24000,اهرم,39488,1405-04-31,6,57525,577,2.310539e+10,...,1,21103883377941566,17914401175772326,15488.0,13849.0,0.0,1.645333,ITM,0.267023,0.968492


In [ ]:
# =====================================================================
# سلول نهایی: دانلود، پردازش و ذخیره محاسبات Volatility و Delta در اکسل
# =====================================================================

# --- ۱. تابع دانلود سابقه قیمت و محاسبه نوسان تاریخی سالانه ---
def download_and_calculate_volatility(stock_id: str, window_size: int) -> float:
    """
    دانلود مستقیم سابقه قیمت پایانی از CDN بورس بر اساس ساختار واقعی JSON
    """
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    url = f'https://cdn.tsetmc.com/api/ClosingPrice/GetClosingPriceDailyList/{stock_id}/0'
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        # دسترسی به آرایه اصلی قیمت‌های روزانه طبق نمونه دیتای شما
        daily_list = data.get('closingPriceDaily', [])
        if not daily_list:
            return 0.45  # پیش‌فرض در صورت عدم وجود داده
        
        # ساخت دیتافریم سابقه قیمت
        df_hist = pd.DataFrame(daily_list)
        
        # تبدیل ستون dEven (مثال شما: 20260715) به فرمت تاریخ پایتون
        df_hist['dEven'] = df_hist['dEven'].astype(str)
        df_hist['date'] = pd.to_datetime(df_hist['dEven'], format='%Y%m%d')
        df_hist = df_hist.sort_values('date').reset_index(drop=True)
        
        # استخراج قیمت پایانی (ستون pClosing بر اساس فرمت JSON ارسالی شما)
        df_hist['pClosing'] = pd.to_numeric(df_hist['pClosing'], errors='coerce')
        df_hist.dropna(subset=['pClosing'], inplace=True)
        
        if len(df_hist) < 10:
            return 0.45
        
        # ===== استفاده از پنجره مشخص =====
        df_window = df_hist.tail(window_size).copy()

        # محاسبه بازده لگاریتمی روزانه بر اساس pClosing
        df_window['log_return'] = np.log(df_window['pClosing'] / df_window['pClosing'].shift(1))
        daily_std = df_window['log_return'].std()
        
        if pd.isna(daily_std) or daily_std == 0:
            return 0.45
            
        # سالانه‌سازی انحراف معیار روزانه (مبنای ۲۲۰ روز معاملاتی سال در ایران)
        return round(float(daily_std * np.sqrt(220)), 4)
        
    except Exception as e:
        # در صورت هرگونه خطا، مقدار پیش‌فرض را برمی‌گرداند تا فرآیند متوقف نشود
        return 0.45


# --- ۳. شروع پردازش دیتای اصلی بازار ---

# کپی از دیتای خروجی DataCleaner شما (فقط اختیارهای خرید)
df_calls = df_final[(df_final['Type'].apply(lambda x: x.name == 'CALL'))].copy()

df_calls['Volatility'] = 0.45

# حلقه بهینه شده با groupby بر روی شناسه بومی بورس ایران
new_records = []
for underlying_ticker, group in df_calls.groupby('UnderlyingTicker'):
    # دریافت کد دارایی پایه از ستون مشخص شده در دیتافریم شما
    underlying_id = group['InstrumentCode-UA'].iloc[0]

    # پنجره بهینه بر اساس حداکثر سررسید
    window_size = max(30, int(group['DaysToMaturity'].max()))
    
    # محاسبه نوسان تاریخی
    hv = download_and_calculate_volatility(str(underlying_id), window_size)
    
    new_records.append({
        'UnderlyingTicker': underlying_ticker,
        'InstrumentCode-UA': underlying_id,
        'WindowSize': window_size,
        'CalculationDate': datetime.now().strftime('%Y-%m-%d'),
        'DaysToMaturity_Ref': int(group['DaysToMaturity'].mean()),
        'Volatility': hv,})
    
    time.sleep(random.uniform(0.1, 1.0))

df_volatility = pd.DataFrame(new_records)

excel_name = "daily_market_volatility.xlsx"
df_volatility.to_excel(excel_name, index=False)


# نمایش ۱۰ ردیف اول جهت بازبینی سریع نتایج در نوت‌بوک
display(df_volatility.head(3))

,UnderlyingTicker,InstrumentCode-UA,WindowSize,CalculationDate,DaysToMaturity_Ref,Volatility
0,اطلس,11427939669935844,30,2026-07-16,17,0.3736
1,اهرم,17914401175772326,62,2026-07-16,32,0.3912
2,تاصيكو,23293437377896568,76,2026-07-16,42,0.6406
3,توان,41927452991671109,31,2026-07-16,11,0.5092
4,جوانه كوچك,67455383896188985,62,2026-07-16,62,0.2975


In [34]:
def covered_call_simple(ticker, premium_call, stock_price, contract_size, days):

    # سود حق‌بیمه
    premium_profit = premium_call * contract_size
    
    # سود کل در قیمت فعلی
    total_profit = premium_profit
    
    # سرمایه اولیه (cost basis)
    cost_basis = stock_price * contract_size

    # درصد بازده
    profit_percent = round((total_profit / cost_basis) * 100, 2)
    monthly_return = round(profit_percent * (30 / days), 2)
    
    # قیمت سربهسر (قیمتی که از آن به بعد ضرر می‌کنیم)
    break_even_price = stock_price - premium_call
    
    # درصد افت تا سربهسر (همان عددی که می‌خواهیم)
    drop_percent = round(((stock_price - break_even_price) / stock_price) * 100, 2)
    
    return {
    'net_profit': total_profit,
    'monthly_return': monthly_return,
    'break_even_price': break_even_price,
    'max_drop_percent': drop_percent}


In [35]:
def covered_call_with_fees(ticker, premium_call, stock_price, strike_price, contract_size,
                           opt_sell_commission, stock_buy_commission, exercise_fee_rate, exercise_tax_rate, days):

    # ====================== 1. کارمزدهای ورود (همیشه اعمال می‌شوند) ======================
    # کارمزد فروش اختیار
    option_fee = -round(premium_call * contract_size * opt_sell_commission, 0)
    # کارمزد خرید سهام
    stock_buy_fee = -round(stock_price * contract_size * stock_buy_commission, 0)
     # کل کارمزد ورود به استراتژی
    entry_fees = option_fee + stock_buy_fee

    # ====================== 2. کارمزدهای خروج/اعمال  ======================
    exercise_fee = -round(strike_price * contract_size * exercise_fee_rate, 0)
    # مالیات اعمال
    exercise_tax = -round(strike_price * contract_size * exercise_tax_rate, 0)

    # ====================== 3. مبالغ اصلی ======================
    premium_received = premium_call * contract_size     # دریافتی از فروش اختیار
    stock_cost = -stock_price * contract_size           # ارزش خرید سهام

    # ====================== 4. سرمایه اولیه خالص ======================
    # سرمایه خالص درگیر (جریان نقدی اولیه)
    net_investment = stock_cost + premium_received + entry_fees

    # تحویل سهام در قیمت اعمال (دریافت وجه)
    strike_received = strike_price * contract_size

    # تحویل سهام در قیمت اعمال (دریافت وجه خالص)
    net_received = strike_received + exercise_fee + exercise_tax

    # ====================== 5. سود خالص ======================
    net_profit = net_received + net_investment

    # ====================== 6. درصد بازده ======================
    profit_percent = (round((net_profit / abs(net_investment)) * 100, 2) if net_investment != 0 else 0)
    monthly_return = round(profit_percent * (30 / days), 2)

    # ====================== 7. قیمت سربه‌سر (فقط سناریوی اصلی) ======================
    downside_protection = premium_received + entry_fees + exercise_fee + exercise_tax
    
    # قیمتی که در آن سرمایه اولیه جبران شود
    break_even_price = round(stock_price - (downside_protection / contract_size), 0)

    # ====================== 8. درصد افت مجاز (فقط سناریوی اصلی) ======================
    max_drop_percent = round(((stock_price - break_even_price) / stock_price) * 100, 2)

    return {
    'net_profit': net_profit,
    'monthly_return': monthly_return,
    'break_even_price': break_even_price,
    'max_drop_percent': max_drop_percent}

In [36]:
filter_option = df_final[
    (df_final['DaysToMaturity'] > 2.0) &
    (df_final['Type'].apply(lambda x: x.name == 'CALL'))].copy()

EXCLUDED_UNDERLYING = ['اهرم']
EXCLUDED_NAME_PATTERN = ['1405/04', '1405-04']
exclude_mask = (
    (filter_option['UnderlyingTicker'].isin(EXCLUDED_UNDERLYING)) & 
    (filter_option['Name'].str.contains('|'.join(EXCLUDED_NAME_PATTERN), na=False)))

filter_option = filter_option[~exclude_mask].copy()

# filter_option = filter_option[filter_option['UnderlyingTicker'].isin(EXCLUDED_UNDERLYING)]

from config import EXERCISE_TAX_RATE, get_symbol_market, get_symbol_kind, get_commission_rate, get_exercise_fee_rate

results = []
results_fee = []
for underlying_symbol, group in filter_option.groupby('UnderlyingTicker'):
    market = get_symbol_market(underlying_symbol)
    kind = get_symbol_kind(underlying_symbol)

    opt_sell_commission = get_commission_rate(market, 'option', False)
    stock_buy_commission = get_commission_rate(market, kind, True)
    exercise_fee_rate = get_exercise_fee_rate(market, kind)
    exercise_tax_rate = EXERCISE_TAX_RATE

    for index, item in group.iterrows():
        # استخراج اطلاعات مورد نیاز
        ticker = item['Ticker']
        strike_price = item['StrikePrice']
        premium_call = item['BidPrice']
        stock_price = item['UnderlyingPrice']
        contract_size = item['ContractSize']
        days = item['DaysToMaturity']
        
        # محاسبات با کارمزد
        results_with_fees = covered_call_with_fees(
            ticker, premium_call, stock_price, strike_price, contract_size,
            opt_sell_commission, stock_buy_commission,
            exercise_fee_rate, exercise_tax_rate, days)
        # محاسبات بدون کارمزد
        results_withoth_fees = covered_call_simple( 
            ticker, premium_call, 
            stock_price, contract_size, days)
        
        # ذخیره نتایج در دیکشنری
        results_fee.append({
            'underlying': underlying_symbol,
            'option_symbol': ticker,
            'strike': strike_price,
            'premium': round(premium_call, 0),
            'stock_price': round(stock_price, 0),
            'net_profit': results_with_fees['net_profit'],
            'monthly_return_%': results_with_fees['monthly_return'],
            'break_even_price': results_with_fees['break_even_price'],
            'max_drop_%': results_with_fees['max_drop_percent'],
            'status': getattr(item['OptionStatus'], 'value', item['OptionStatus']),
            'days_to_maturity': days,
            'volume': int(item.get('Volume', 0))
        })
        results.append({
            'underlying': underlying_symbol,
            'option_symbol': ticker,
            'strike': strike_price,
            'premium': round(premium_call, 0),
            'stock_price': round(stock_price, 0),
            'net_profit': results_withoth_fees['net_profit'],
            'monthly_return_%': results_withoth_fees['monthly_return'],
            'break_even_price': results_withoth_fees['break_even_price'],
            'max_drop_%': results_withoth_fees['max_drop_percent'],
            'status': getattr(item['OptionStatus'], 'value', item['OptionStatus']),
            'days_to_maturity': days,
            'volume': int(item.get('Volume', 0))
        })

result_df_fee = pd.DataFrame(results_fee)
result_df = pd.DataFrame(results)

In [41]:
# =====================================================================
# سلول نهایی: ادغام نوسانات، محاسبه دلتا و امتیازدهی روی خروجی کارمزدها
# =====================================================================

# --- ۱. فرمول محاسباتی سبک دلتا بلک-شولز ---
def calculate_black_scholes_delta(S, K, T, r, sigma):
    """
    محاسبه دلتای اختیار خرید (Call Option Delta)
    """
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.50  # مقدار مرزی فرضی برای آپشن‌های نزدیک سررسید
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    return round(float(np.clip(norm.cdf(d1), 0.0, 1.0)), 4)


# --- ۲. تابع امتیازدهی هوشمند هماهنگ با ستون‌های دیتافریم جدید شما ---
def score_covered_call_adapted(row, hv_col='Volatility', delta_col='Delta'):
    """
    سیستم امتیازدهی استراتژی Covered Call منطبق با ساختار ستون‌های سلول قبل
    """
    monthly_return = row['monthly_return_%']
    max_drop = row['max_drop_%']
    dte = row['days_to_maturity']
    stock_price = row['stock_price']
    strike = row['strike']
    hv = row[hv_col]
    delta = row[delta_col]

    # امتیازدهی بخش‌های مختلف
    return_score = np.clip(monthly_return / 12.0, 0.0, 1.0)
    protection_score = np.clip(max_drop / 25.0, 0.0, 1.0)

    # حاشیه امنیت متناسب با نوسان سهم (Expected Move)
    expected_move = 1.5 * hv * np.sqrt(dte / 365.0) * 100
    downside_score = np.clip(max_drop / expected_move, 0.0, 1.0) if expected_move > 0 else 1.0

    # امتیاز جهت‌گیری بازار (Delta Score)
    net_delta = abs(1.0 - delta)
    delta_score = np.exp(-2.0 * net_delta)

    # جریمه نمایی برای گزینه‌های عمیقاً درون سود (ITM)
    moneyness = (stock_price - strike) / stock_price if stock_price > 0 else 0
    itm_penalty = (1.0 - np.exp(-6.0 * moneyness)) if moneyness > 0 else 0.0

    final_score = (
        0.30 * return_score +
        0.25 * protection_score +
        0.25 * downside_score +
        0.10 * delta_score -
        0.10 * itm_penalty)
    return round(np.clip(final_score * 100, 0, 100), 2)


# --- ۳. لود داده‌های نوسان ذخیره شده روزانه ---
vol_file = "daily_market_volatility.xlsx"
if os.path.exists(vol_file):
    df_vol = pd.read_excel(vol_file)
    # هماهنگ‌سازی کلید واژه ادغام با نام ستون سلول شما ('underlying')
    df_vol_subset = df_vol[['UnderlyingTicker', 'Volatility']].rename(columns={'UnderlyingTicker': 'underlying'})
else:
    df_vol_subset = pd.DataFrame(columns=['underlying', 'Volatility'])


# --- ۴. تابع کمکی برای پردازش، محاسبه دلتا و امتیازدهی دیتای ورودی شما ---
def process_and_score_dataset(df_input, df_volatility, r_free=0.25):
    # الف. ادغام با جدول نوسان بر اساس ستون underlying
    df_merged = pd.merge(df_input, df_volatility, on='underlying', how='left')
    df_merged['Volatility'] = df_merged['Volatility'].fillna(0.45) # جایگزینی نوسان‌های خالی با مقدار پیش‌فرض
    
    # ب. محاسبه دلتا به صورت نظیر به نظیر برای تک‌تک آپشن‌ها
    df_merged['Delta'] = df_merged.apply(
        lambda r: calculate_black_scholes_delta(
            S=r['stock_price'],
            K=r['strike'],
            T=r['days_to_maturity'] / 365.0,
            r=r_free,
            sigma=r['Volatility']), axis=1)
    
    # ج. اعمال تابع امتیازدهی
    df_merged['score'] = df_merged.apply(score_covered_call_adapted, axis=1)
    
    # د. مرتب‌سازی صعودی بر اساس بالاترین امتیاز کسب شده
    return df_merged.sort_values(by='score', ascending=False).reset_index(drop=True)


# --- ۵. اجرای خط لوله پردازش برای هر دو حالت (با کارمزد و بدون کارمزد) ---
r_free_rate = 0.25  # نرخ بدون ریسک فرضی

# پردازش دیتافریم محاسبات با کارمزد
scored_df_fee = process_and_score_dataset(result_df_fee, df_vol_subset, r_free=r_free_rate)

# --- 6. نمایش چند ردیف برتر نتایج با اعمال کارمزد جهت بررسی وضعیت نهایی ---
display(
    scored_df_fee[[
        'option_symbol', 'underlying', 'stock_price', 'strike', 
        'days_to_maturity', 'monthly_return_%', 'max_drop_%', 
        'Volatility', 'Delta', 'score'
    ]].head(3))

,option_symbol,underlying,stock_price,strike,days_to_maturity,monthly_return_%,max_drop_%,Volatility,Delta,score
0,ضملي7065,فملي,19800,16000,76,5.81,29.51,0.2556,0.9901,67.49
1,ضستا5039,شستا,2086,1200,20,5.29,44.44,0.2629,1.0000,64.01
2,ضهرم6043,اهرم,39488,34000,62,5.70,22.93,0.3912,0.8983,63.39


In [42]:
scored_df_fee['dte_factor'] = (scored_df_fee['days_to_maturity'] / 30) ** 0.5
scored_df_fee['dte_factor'] = scored_df_fee['dte_factor'].clip(lower=0.3, upper=2.5)

scored_df_fee['max_drop_threshold'] = 10.0 * scored_df_fee['dte_factor']

filtered_df_fee = scored_df_fee[scored_df_fee['max_drop_%'] >= scored_df_fee['max_drop_threshold']].copy()
filtered_df_fee = filtered_df_fee.sort_values(by='monthly_return_%', ascending=False).reset_index(drop=True)
filtered_df_fee.to_excel('filtered_df_fee.xlsx', index='False')

In [ ]:
import pandas as pd
df=pd.DataFrame(result_df)
df.to_excel('covered call.xlsx',index='False')
df

,long stock,short_call,total_payoff,coveredcall_percent
0,خودرو,ضخود5037,44000,7.68
1,خودرو,ضخود5038,46000,8.03
2,خودرو,ضخود5039,59000,10.30
3,خودرو,ضخود5040,79000,13.79
4,خودرو,ضخود5041,143000,24.96
5,خودرو,ضخود5042,232000,40.49
6,خودرو,ضخود5043,343000,59.86
7,خودرو,ضخود5044,428000,74.69
8,خودرو,ضخود5045,528000,92.15
9,خودرو,ضخود6037,54000,9.42
